# ERCP Baseline Pipeline (Mock Model)

Dataset root: `training/dataset`  
Classes: `Biliary_Leaks`, `Lithiasis`, `Stricture`, `Normal`

This notebook provides a reusable Torch-compatible baseline with:
- configurable, modular preprocessing and augmentation pipelines
- class-imbalance strategy switching
- evaluation on all validation samples (DenseNet-style metrics)
- Grad-CAM visualization artifact generation


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import Callable, Dict, List, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns


## Configuration
Define dataset paths, class labels, and run modes.


In [ ]:
DATASET_ROOT = Path('dataset')
ARTIFACT_DIR = Path('models')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['Biliary_Leaks', 'Lithiasis', 'Stricture', 'Normal']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

class ImbalanceMode(str, Enum):
    NORMAL = 'normal'
    WEIGHTED_LOSS = 'weighted_loss'
    UNIFORM_SAMPLING = 'uniform_sampling'


## Pipeline Configuration
Set modular preprocessing, augmentation, and run-time configuration objects.


In [ ]:
@dataclass
class PreprocessStep:
    name: str
    enabled: bool = True

@dataclass
class AugmentStep:
    name: str
    enabled: bool = True
    prob: float = 1.0

@dataclass
class PreprocessConfig:
    resize: Tuple[int, int] = (224, 224)
    normalize_mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    normalize_std: Tuple[float, float, float] = (0.229, 0.224, 0.225)
    clahe_clip_limit: float = 2.0
    clahe_tile_grid: Tuple[int, int] = (8, 8)
    steps: Tuple[PreprocessStep, ...] = (
        PreprocessStep('clahe', enabled=True),
        PreprocessStep('resize', enabled=True),
    )

@dataclass
class AugmentConfig:
    rotation_deg: float = 10.0
    zoom_factor: float = 1.08
    brightness_delta: float = 0.1
    blur_kernel: int = 3
    preserve_border: bool = True
    steps: Tuple[AugmentStep, ...] = (
        AugmentStep('brightness', enabled=True, prob=0.7),
        AugmentStep('blur', enabled=True, prob=0.4),
        AugmentStep('rotation', enabled=True, prob=0.7),
        AugmentStep('zoom', enabled=True, prob=0.6),
    )

@dataclass
class RunConfig:
    imbalance_mode: ImbalanceMode = ImbalanceMode.NORMAL
    batch_size: int = 8
    num_workers: int = 0
    lr: float = 1e-3
    epochs: int = 1
    use_gradcam: bool = True


In [ ]:
PREPROCESS_REGISTRY: Dict[str, Callable[[np.ndarray, PreprocessConfig], np.ndarray]] = {}

def register_preprocess(name: str):
    def deco(fn):
        PREPROCESS_REGISTRY[name] = fn
        return fn
    return deco

@register_preprocess('clahe')
def preprocess_clahe(img_bgr: np.ndarray, cfg: PreprocessConfig) -> np.ndarray:
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=cfg.clahe_clip_limit, tileGridSize=cfg.clahe_tile_grid)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

@register_preprocess('resize')
def preprocess_resize(img_bgr: np.ndarray, cfg: PreprocessConfig) -> np.ndarray:
    return cv2.resize(img_bgr, cfg.resize, interpolation=cv2.INTER_AREA)

def apply_preprocessing_pipeline(img_bgr: np.ndarray, cfg: PreprocessConfig) -> np.ndarray:
    out = img_bgr
    for step in cfg.steps:
        if not step.enabled:
            continue
        fn = PREPROCESS_REGISTRY.get(step.name)
        if fn is None:
            raise ValueError(f'Unknown preprocessing step: {step.name}')
        out = fn(out, cfg)
    return out


## Image Preprocessing and Augmentation
Register reusable preprocessing steps and augmentation operations.


In [ ]:
def detect_content_bbox(img_bgr: np.ndarray, threshold: int = 8):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(gray > threshold)
    if len(xs) == 0 or len(ys) == 0:
        return 0, 0, img_bgr.shape[1], img_bgr.shape[0]
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return x0, y0, x1, y1

def border_preserving_zoom(img_bgr: np.ndarray, zoom_factor: float) -> np.ndarray:
    if zoom_factor <= 1.0:
        return img_bgr
    h, w = img_bgr.shape[:2]
    x0, y0, x1, y1 = detect_content_bbox(img_bgr)
    roi = img_bgr[y0:y1+1, x0:x1+1]
    rh, rw = roi.shape[:2]
    zh, zw = max(1, int(rh / zoom_factor)), max(1, int(rw / zoom_factor))
    cy, cx = rh // 2, rw // 2
    sy, sx = max(0, cy - zh // 2), max(0, cx - zw // 2)
    crop = roi[sy:sy+zh, sx:sx+zw]
    zoomed = cv2.resize(crop, (rw, rh), interpolation=cv2.INTER_LINEAR)
    out = img_bgr.copy()
    out[y0:y1+1, x0:x1+1] = zoomed
    return out

def augment_brightness(img_bgr: np.ndarray, cfg: AugmentConfig) -> np.ndarray:
    alpha = 1.0 + np.random.uniform(-cfg.brightness_delta, cfg.brightness_delta)
    return cv2.convertScaleAbs(img_bgr, alpha=alpha, beta=0)

def augment_blur(img_bgr: np.ndarray, cfg: AugmentConfig) -> np.ndarray:
    k = max(1, int(cfg.blur_kernel))
    if k % 2 == 0:
        k += 1
    return cv2.GaussianBlur(img_bgr, (k, k), 0)

def augment_rotation(img_bgr: np.ndarray, cfg: AugmentConfig) -> np.ndarray:
    ang = np.random.uniform(-cfg.rotation_deg, cfg.rotation_deg)
    h, w = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    return cv2.warpAffine(img_bgr, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def augment_zoom(img_bgr: np.ndarray, cfg: AugmentConfig) -> np.ndarray:
    if cfg.preserve_border:
        return border_preserving_zoom(img_bgr, cfg.zoom_factor)
    return img_bgr

AUGMENT_REGISTRY: Dict[str, Callable[[np.ndarray, AugmentConfig], np.ndarray]] = {
    'brightness': augment_brightness,
    'blur': augment_blur,
    'rotation': augment_rotation,
    'zoom': augment_zoom,
}

def apply_augmentation_pipeline(img_bgr: np.ndarray, cfg: AugmentConfig) -> np.ndarray:
    out = img_bgr.copy()
    for step in cfg.steps:
        if not step.enabled:
            continue
        if np.random.rand() > float(step.prob):
            continue
        fn = AUGMENT_REGISTRY.get(step.name)
        if fn is None:
            raise ValueError(f'Unknown augmentation step: {step.name}')
        out = fn(out, cfg)
    return out


In [ ]:
def collect_split_samples(split: str) -> List[Tuple[Path, int]]:
    split_dir = DATASET_ROOT / split
    if not split_dir.exists():
        return []
    samples: List[Tuple[Path, int]] = []
    for cname in CLASSES:
        cdir = split_dir / cname
        if not cdir.exists():
            continue
        for p in cdir.glob('*.png'):
            samples.append((p, CLASS_TO_IDX[cname]))
    return samples

train_samples = collect_split_samples('train')
val_samples = collect_split_samples('val')  # strict protocol; may be empty
test_samples = collect_split_samples('test')
print('train/val/test:', len(train_samples), len(val_samples), len(test_samples))


In [ ]:
class ERCPDataset(Dataset):
    def __init__(self, samples, pre_cfg: PreprocessConfig, aug_cfg: AugmentConfig | None = None, train: bool = False):
        self.samples = samples
        self.pre_cfg = pre_cfg
        self.aug_cfg = aug_cfg
        self.train = train
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=pre_cfg.normalize_mean, std=pre_cfg.normalize_std),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))
        if img is None:
            # unreadable image fallback
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        # Apply selected preprocessing steps in configured order.
        img = apply_preprocessing_pipeline(img, self.pre_cfg)

        # Apply selected augmentation steps with per-step probability during training only.
        if self.train and self.aug_cfg is not None:
            img = apply_augmentation_pipeline(img, self.aug_cfg)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(img)
        x = self.to_tensor(pil)
        return x, torch.tensor(label, dtype=torch.long), str(path)


## Model Definition
Define the baseline mock model architecture.


In [ ]:
class MockModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        f = self.features(x).flatten(1)
        return self.classifier(f)


## Data Loaders and Imbalance Strategy
Build train loader and imbalance-aware sampling or loss behavior.


In [ ]:
def build_train_loader(samples, pre_cfg, aug_cfg, run_cfg):
    ds = ERCPDataset(samples, pre_cfg, aug_cfg=aug_cfg, train=True)
    if run_cfg.imbalance_mode == ImbalanceMode.UNIFORM_SAMPLING:
        labels = np.array([y for _, y in samples])
        counts = np.bincount(labels, minlength=len(CLASSES)) + 1e-6
        w = 1.0 / counts
        sw = np.array([w[y] for y in labels], dtype=np.float32)
        sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
        return DataLoader(ds, batch_size=run_cfg.batch_size, sampler=sampler, num_workers=run_cfg.num_workers)
    return DataLoader(ds, batch_size=run_cfg.batch_size, shuffle=True, num_workers=run_cfg.num_workers)

def class_weights_from_samples(samples):
    labels = np.array([y for _, y in samples])
    counts = np.bincount(labels, minlength=len(CLASSES)) + 1e-6
    w = counts.sum() / counts
    return torch.tensor(w, dtype=torch.float32)


## Model Training and Evaluation
Train the model and evaluate on all validation samples with metrics and confusion matrix.


In [ ]:
def evaluate_model(data_loader: DataLoader, model: nn.Module, device: str):
    predictions = []
    actual_values = []
    model.eval()

    with torch.no_grad():
        for inputs, labels, _ in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            y_pred = model(inputs)
            y_pred = y_pred.detach().cpu().numpy()
            actual = labels.cpu().numpy()
            y_pred = np.argmax(y_pred, axis=1)
            predictions.append(y_pred.reshape((-1, 1)))
            actual_values.append(actual.reshape((-1, 1)))

    if not predictions:
        return np.array([]), np.array([])

    predictions = np.vstack(predictions)
    actual_values = np.vstack(actual_values)

    f1 = f1_score(actual_values, predictions, average='macro', zero_division=0)
    acc = accuracy_score(actual_values, predictions)
    print(f'Validation Accuracy: {acc:0.4f}')
    print(f'Validation F1 (macro): {f1:0.4f}')
    return actual_values, predictions


def display_confusion_matrix(cm, list_classes, filename):
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        cm,
        annot=True,
        xticklabels=list_classes,
        yticklabels=list_classes,
        annot_kws={'size': 10},
        fmt='g',
        linewidths=.5,
    )
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()


class Trainer:
    def __init__(self, train_samples, val_samples):
        self.train_samples = train_samples
        self.val_samples = val_samples

    def train(self, model: nn.Module, optimizer, num_iterations: int | None, epochs: int | None,
              save_dir: Path, model_name: str, preprocess_config: PreprocessConfig,
              augmentation_config: AugmentConfig, run_config: RunConfig):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = model.to(device)

        train_loader = build_train_loader(self.train_samples, preprocess_config, augmentation_config, run_config)
        cw = class_weights_from_samples(self.train_samples).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw if run_config.imbalance_mode == ImbalanceMode.WEIGHTED_LOSS else None)

        total_epochs = int(epochs if epochs is not None else max(1, int(num_iterations or 1)))
        model.train()
        for _ in range(total_epochs):
            i = 0
            for x, y, _ in train_loader:
                print(str(i) + " out of " + str(len(train_loader)) + "\r")
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()
                i=i+1
                if i == 20:
                    break

        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), save_dir / model_name)
        return model

    def evaluate(self, trained_model: nn.Module, eval_loader: DataLoader, run_config: RunConfig):
        device = next(trained_model.parameters()).device
        actual_values, predictions = evaluate_model(eval_loader, trained_model, device)

        metrics = {
            'per_class': {c: 0.0 for c in CLASSES},
            'aggregate': {'accuracy': 0.0, 'f1_macro': 0.0},
            'artifacts': {'confusion_matrix': None, 'classification_report': None},
        }

        if actual_values.size > 0:
            report_text = classification_report(actual_values, predictions, target_names=CLASSES, digits=4, zero_division=0)
            cm = confusion_matrix(actual_values, predictions)

            acc = accuracy_score(actual_values, predictions)
            f1 = f1_score(actual_values, predictions, average='macro', zero_division=0)
            metrics['aggregate']['accuracy'] = float(acc)
            metrics['aggregate']['f1_macro'] = float(f1)

            for i, c in enumerate(CLASSES):
                cls_mask = (actual_values.flatten() == i)
                denom = max(1, int(cls_mask.sum()))
                metrics['per_class'][c] = float((predictions.flatten()[cls_mask] == i).sum() / denom)

            cm_path = ARTIFACT_DIR / 'baseline_mock_model_cm.png'
            report_path = ARTIFACT_DIR / 'baseline_mock_model_report.txt'

            # Display confusion matrix in output and persist image artifact.
            display_confusion_matrix(cm, CLASSES, str(cm_path))

            # Print report text to notebook output and persist text artifact.
            print('Classification Report:')
            print(report_text)

            report_lines = [
                'ERCP Validation Evaluation Report',
                '=' * 36,
                f"Accuracy: {metrics['aggregate']['accuracy']:.4f}",
                f"F1 Macro: {metrics['aggregate']['f1_macro']:.4f}",
                '',
                'Per-class Accuracy:',
            ]
            report_lines.extend([f"- {k}: {v:.4f}" for k, v in metrics['per_class'].items()])
            report_lines.extend([
                '',
                'Classification Report:',
                str(report_text),
                '',
                'Confusion Matrix:',
                str(np.array2string(cm)),
                '',
            ])
            report_path.write_text(''.join(report_lines))

            metrics['artifacts']['confusion_matrix'] = str(cm_path)
            metrics['artifacts']['classification_report'] = str(report_path)

        return metrics


## Grad-CAM Visualization
Generate class-activation maps to inspect model attention regions.


In [ ]:
def gradcam_overlay(model: nn.Module, x: torch.Tensor, target_class: int):
    model.eval()
    device = next(model.parameters()).device
    feats, grads = {}, {}

    def fw_hook(_, __, out):
        feats['v'] = out

    def bw_hook(_, gin, gout):
        grads['v'] = gout[0]

    h1 = model.features[-3].register_forward_hook(fw_hook)
    h2 = model.features[-3].register_full_backward_hook(bw_hook)

    x = x.unsqueeze(0).to(device)
    logits = model(x)
    score = logits[0, target_class]
    model.zero_grad()
    score.backward()

    fmap = feats['v'][0]
    grad = grads['v'][0]
    w = grad.mean(dim=(1,2), keepdim=True)
    cam = (w * fmap).sum(0).detach().cpu().numpy()
    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)

    h1.remove(); h2.remove()
    return cam


## Smoke Run
Execute an end-to-end sanity check of training, evaluation, and Grad-CAM outputs.


In [ ]:
# Smoke run
pre_cfg = PreprocessConfig()
aug_cfg = AugmentConfig()
run_cfg = RunConfig(imbalance_mode=ImbalanceMode.NORMAL, epochs=1)

# Proceed only when both train and val splits are available.
if len(train_samples) > 0 and len(val_samples) > 0:
    model = MockModel(num_classes=len(CLASSES))
    optimizer = torch.optim.Adam(model.parameters(), lr=run_cfg.lr)
    trainer = Trainer(train_samples=train_samples, val_samples=val_samples)

    trained_model = trainer.train(
        model=model,
        optimizer=optimizer,
        num_iterations=None,
        epochs=run_cfg.epochs,
        save_dir=ARTIFACT_DIR,
        model_name='baseline_mock_model.pth',
        preprocess_config=pre_cfg,
        augmentation_config=aug_cfg,
        run_config=run_cfg,
    )

    val_loader = DataLoader(ERCPDataset(val_samples, pre_cfg, train=False), batch_size=run_cfg.batch_size, shuffle=False)
    metrics = trainer.evaluate(trained_model=trained_model, eval_loader=val_loader, run_config=run_cfg)
    print('Per-class validation metrics:', metrics['per_class'])
    print('Aggregate metrics:', metrics['aggregate'])

    # Generate one Grad-CAM artifact from validation samples.
    if run_cfg.use_gradcam:
        ds = ERCPDataset(val_samples, pre_cfg, train=False)
        x, y, p = ds[0]
        cam = gradcam_overlay(trained_model, x, int(y))
        np.save(ARTIFACT_DIR / 'gradcam_sample.npy', cam)
        print('Saved Grad-CAM:', ARTIFACT_DIR / 'gradcam_sample.npy', 'for', p)
else:
    print('Skipped smoke run: train and/or val split is empty.')
